# 🤖 Notebook 04 — Model Training
## Bagian 4: Training Model ML + Experiment Runner

**Subset Ringan:**
- Feature Extractors: Glove, FastText, Word2Vec
- Models: Decision Tree, Random Forest, XGBoost

**Total kombinasi:** 9 kombinasi

## Setup

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import scipy.sparse as sp

from src.experiment_runner import run_experiments

## Automated Experiment Runner

Jalankan kombinasi model ringan dengan Word2Vec.

In [2]:
# Load dataset yang sudah dilabeli dan dipreprocessing
df = pd.read_csv('../data/processed/reviews_prepared_new.csv')
print(f"📊 Dataset: {df.shape[0]} baris")

# Pastikan kolom yang diperlukan ada
assert 'review_clean' in df.columns, "❌ Jalankan Notebook 02 terlebih dahulu!"
assert 'sentiment_encoded' in df.columns, "❌ Jalankan Notebook 02 terlebih dahulu!"

📊 Dataset: 324 baris


In [3]:
# Jalankan experiment runner (Menggunakan dataset balanced dari filter 2024-2026)
print("\n🚀 Menjalankan Experiment Runner (Balanced Dataset 2024-2026)...")
print("   Extractors: Word2Vec, FastText, TF-IDF, GloVe")
print("   Models: Decision Tree, Random Forest, XGBoost")
print("   ⏱️  Estimasi waktu: 2-3 menit\n")

# Gunakan data yang sudah diseimbangkan dan di-filter tahun 2024-2026
df_balanced = pd.read_csv('../data/processed/reviews_prepared.csv')

print(f"📊 Dataset Info:")
print(f"   Total baris: {len(df_balanced)}")
print(f"   Sentimen distribution: {df_balanced['sentiment'].value_counts().to_dict()}\n")

comparison_df = run_experiments(
    df_balanced,
    text_col='review_clean', 
    label_col='sentiment_encoded',
    subset='priority',
    test_size=0.2,
    random_state=42,
    n_iter=5,
    cv=3,                   
    save_features=True,
    save_dir='../results'
)


🚀 Menjalankan Experiment Runner (Balanced Dataset 2024-2026)...
   Extractors: Word2Vec, FastText, TF-IDF, GloVe
   Models: Decision Tree, Random Forest, XGBoost
   ⏱️  Estimasi waktu: 2-3 menit

📊 Dataset Info:
   Total baris: 609
   Sentimen distribution: {'Positif': 203, 'Netral': 203, 'Negatif': 203}


🚀 EXPERIMENT RUNNER — Klasifikasi Sentimen CoreTax

📊 Dataset split:
   Train: 487 | Test: 122
   Train distribution: [163 162 162]
   Test distribution:  [40 41 41]

🔧 Extractors: ['GloVe', 'FastText', 'Word2Vec', 'TF-IDF']
🤖 Models: ['Decision Tree', 'Random Forest', 'XGBoost']

📈 Total kombinasi valid: 12
----------------------------------------------------------------------

📦 Feature Extractor: GloVe
   GloVe: Loading vectors dari ../data/embeddings/cc.id.300.vec...
   GloVe: vocab loaded=200000
   GloVe: OOV rate = 183/1372 (13.3%)
   GloVe: shape=(487, 300)
   💾 Features saved: glove
   ⏱️  Extraction time: 7.9s

   [1/12] 🤖 GloVe + Decision Tree
      ✅ W-F1=0.5758 | M-F1=0.

KeyboardInterrupt: 

In [ ]:
# Tampilkan hasil
if comparison_df is not None:
    print("\n📊 Tabel Komparasi Lengkap:")
    print(comparison_df[['feature_extractor', 'model', 'weighted_f1', 'macro_f1',
                          'accuracy', 'train_time_s']].to_string())

## Ringkasan Model Training

## 🔧 Fix Log: Sparse Array Length Error

**Error yang diperbaiki:**
```
Error: sparse array length is ambiguous; use getnnz() or shape[0]
```

**Root Cause:**
- Scikit-learn RandomizedSearchCV tidak handle sparse matrix dengan baik
- TF-IDF menghasilkan sparse CSR matrix
- Cross-validation splitting pada sparse matrix menyebabkan error

**Solusi Implementasi:**
1. **experiment_runner.py**: Convert semua sparse matrix ke dense sebelum passing ke model.fit()
2. **models.py**: 
   - Ubah error_score dari "raise" menjadi 0.0 di RandomizedSearchCV
   - Simplify predict/predict_proba (tidak perlu conversion lagi)
   - Set needs_dense=False untuk semua model (conversion sudah di experiment_runner)
3. **evaluator.py**: Fix measure_inference_time untuk handle sparse matrix dengan X_test.shape[0]

**Hasil:**
✅ TF-IDF + Decision Tree
✅ TF-IDF + Random Forest  
✅ TF-IDF + XGBoost
Semua berjalan tanpa error!

In [ ]:
if comparison_df is not None and len(comparison_df) > 0:
    print("\n🏆 Top 3 Kombinasi Terbaik:")
    print("=" * 80)
    top3 = comparison_df.head(3)
    for idx, row in top3.iterrows():
        print(f"   #{idx+1}: {row['feature_extractor']} + {row['model']}")
        print(f"         W-F1={row['weighted_f1']:.4f} | M-F1={row['macro_f1']:.4f} | "
              f"Train={row['train_time_s']:.1f}s")

    best = comparison_df.iloc[0]
    print(f"\n🥇 BEST: {best['feature_extractor']} + {best['model']}")
    print(f"   Weighted F1 = {best['weighted_f1']:.4f}")

In [ ]:
print("\n✅ Model training selesai! Lanjut ke Notebook 05 untuk analisis komparatif.")